# Stage 6 - Figures and tables of the paper

**Input:** the tables written by stages 2 to 5 in `data/processed/`
**Output:** the figures of the paper in `figures/` (PNG, 300 dpi), and the tables printed below

This notebook computes nothing new: it only draws what the previous stages saved, so every figure of the paper can be regenerated by running it.

*Tool declaration: this notebook was drafted with Claude (Anthropic), asked to draw the figures listed below from the saved tables.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from helpers import FIGURES, PROCESSED, load_border_crossings, to_km

FIGURES.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# Colours that remain distinguishable for colour-blind readers (Okabe-Ito palette)
BLUE, ORANGE, GREEN, PINK, GREY = "#0072B2", "#E69F00", "#009E73", "#CC79A7", "#7F7F7F"

def save(fig, name):
    """Save a figure in figures/ and show it."""
    fig.savefig(FIGURES / f"{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

crossings = load_border_crossings()
crossings["x_km"], crossings["y_km"] = to_km(crossings["lat"], crossings["lon"])
geneva = crossings[crossings["canton"] == "GE"]
valais = crossings[crossings["canton"] == "VS"]

## Table 1 - Cleaning steps

In [ ]:
pd.read_csv(PROCESSED / "cleaning_log.csv")

## Table 2 - Summary statistics

In [ ]:
sales = pd.read_csv(PROCESSED / "sales_features.csv")
columns = ["price_m2", "surface", "is_house", "new_build", "dist_border_km", "dist_geneva_km",
           "median_income", "second_home_share", "cross_border_share"]
sales[columns].describe().T[["mean", "std", "min", "50%", "max"]].round(2)

## Figure 1 - The sales and the border

In [ ]:
sales["x_km"], sales["y_km"] = to_km(sales["latitude"], sales["longitude"])

fig, ax = plt.subplots(figsize=(8, 6))
points = ax.scatter(sales["x_km"], sales["y_km"], c=sales["price_m2"], s=1, cmap="viridis",
                    vmin=2000, vmax=9000, rasterized=True)
ax.scatter(geneva["x_km"], geneva["y_km"], color="red", marker="x", s=50, label="Crossings into Geneva")
ax.scatter(valais["x_km"], valais["y_km"], color="red", marker="^", s=40, label="Crossings into Valais")
fig.colorbar(points, label="Price per m2 (euros)", extend="both")
ax.set_xlabel("Kilometres east of the reference point")
ax.set_ylabel("Kilometres north of the reference point")
ax.set_title(f"{len(sales):,} home sales in Haute-Savoie, 2021-2025")
ax.set_aspect("equal")
ax.legend(loc="lower left")
save(fig, "fig1_sales_map")

## Figure 2 - Model comparison on unseen municipalities

In [ ]:
comparison = pd.read_csv(PROCESSED / "model_comparison.csv")
models = ["Linear regression", "Random forest", "Gradient boosting"]
specs = ["no border", "any crossing", "Geneva crossing", "cross-border share"]
r2 = comparison.pivot(index="model", columns="specification", values="R2").loc[models, specs]

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.2
for i, (spec, colour) in enumerate(zip(specs, [GREY, ORANGE, BLUE, GREEN])):
    ax.bar(np.arange(len(models)) + (i - 1.5) * width, r2[spec], width, label=spec, color=colour)
ax.set_xticks(np.arange(len(models)))
ax.set_xticklabels(models)
ax.set_ylabel("R2 on unseen municipalities")
ax.set_title("Does a border variable help to predict prices?")
ax.legend(title="Border variable", frameon=False, bbox_to_anchor=(1.0, 1.0), loc="upper left")
save(fig, "fig2_model_comparison")
r2.round(3)

## Figure 3 - Shape of the border premium

In [ ]:
shape = pd.read_csv(PROCESSED / "border_shape.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(shape["dist_geneva_km"], 100 * (np.exp(shape["premium_vs_20km"]) - 1), color=BLUE, linewidth=2)
ax.axhline(0, color=GREY, linewidth=0.8)
ax.set_xlabel("Distance to nearest Geneva crossing (km)")
ax.set_ylabel("Predicted premium vs. 20 km (%)")
ax.set_title("The border premium fades within about 10 km")
save(fig, "fig3_border_shape")

## Figure 4 - The border premium year by year

In [ ]:
by_year = pd.read_csv(PROCESSED / "border_effect_by_year.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(by_year["year"], by_year["premium_2km_vs_20km_pct_mean"],
            yerr=by_year["premium_2km_vs_20km_pct_std"], color=BLUE, marker="o", capsize=4, linewidth=2)
ax.axhline(0, color=GREY, linewidth=0.8)
ax.set_xticks(by_year["year"])
ax.set_ylim(bottom=min(0, ax.get_ylim()[0]))
ax.set_xlabel("Year of sale")
ax.set_ylabel("Premium at 2 km vs. 20 km (%)")
ax.set_title("Border premium by year (bars: spread across the 5 test groups)")
save(fig, "fig4_premium_by_year")
by_year.round(4)

## Figure 5 - Map of the prices the model without border does not explain

In [ ]:
surface = pd.read_csv(PROCESSED / "gp_surface.csv")

fig, ax = plt.subplots(figsize=(9, 6.5))
limit = np.abs(surface["surface_pct"]).quantile(0.99)
cells = ax.scatter(surface["x_km"], surface["y_km"], c=surface["surface_pct"], cmap="RdBu_r",
                   vmin=-limit, vmax=limit, s=14, marker="s", rasterized=True)
ax.scatter(geneva["x_km"], geneva["y_km"], color="black", marker="x", s=50, label="Crossings into Geneva")
ax.scatter(valais["x_km"], valais["y_km"], color="black", marker="^", s=40, label="Crossings into Valais")
fig.colorbar(cells, label="Price vs. prediction of the model without border (%)", extend="both")
ax.set_xlabel("Kilometres east of the reference point")
ax.set_ylabel("Kilometres north of the reference point")
ax.set_title("Local price differences found by the Gaussian process")
ax.set_aspect("equal")
ax.legend(loc="lower left")
save(fig, "fig5_gp_map")

## Table 3 - Unexplained price by distance to Geneva (Gaussian process)

In [ ]:
pd.read_csv(PROCESSED / "gp_by_distance.csv").round(1)